# Why Neural Networks Forget

% TODO: 2-3 sentence hook. What will the reader *see* by the end of this chapter?

In this chapter we train a small network on a sequence of tasks and watch it
forget the earlier ones. No fixes yet, only the problem, measured carefully.

**What you'll learn**

- What catastrophic forgetting is [@mccloskey1989; @french1999]
- How to build the Split-MNIST benchmark
- How to record and read an **accuracy matrix**
- Two standard metrics: average accuracy and backward transfer (BWT) [@lopezpaz2017gem]

## 1. The intuition

% TODO: Explain forgetting in your own words, before any code.
% - An analogy (learning a new phone number and losing the old one?) and where it breaks down
% - Key point: the same weights are shared by every task, and gradient descent on
%   task B only "sees" the task B loss, so nothing stops it from overwriting task A.
% - Contrast with how humans learn (maybe a short note on complementary learning systems)

## 2. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Five tasks, two digits each
TASKS = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

# TODO: tune once everything runs end to end
EPOCHS_PER_TASK = 1
BATCH_SIZE = 128
LR = 1e-3

## 3. Building Split-MNIST

% TODO: Describe the benchmark: MNIST split into 5 tasks of 2 digits each,
% seen one after another. Cite where it comes from.

% TODO: Decide and explain the output head. This choice drives how much forgetting you see:
% - Single head, 10 outputs, original labels  -> class-incremental (dramatic forgetting)
% - Labels remapped to {0, 1} per task        -> domain-incremental
% - Separate head per task                    -> task-incremental (mild forgetting)
% Pick one for this chapter and note that chapter 2 covers all three [@vandeven2022three].

In [ ]:
def make_split_mnist(tasks, root="data"):
    """Return a list of (train_subset, test_subset), one pair per task.

    TODO:
    - load MNIST train and test sets with torchvision (transforms.ToTensor())
    - for each digit pair, keep only the examples whose label is in the pair
    - apply your output-head decision from above (keep or remap labels)
    """
    raise NotImplementedError


# task_data = make_split_mnist(TASKS)

In [ ]:
# TODO: Sanity check. Show a few images per task with their labels,
# and print the number of train/test examples in each task.

## 4. A small model

% TODO: One line on why a plain MLP is enough: forgetting isn't a quirk of a
% particular architecture.

In [ ]:
class MLP(nn.Module):
    # TODO: 784 -> 400 -> 400 -> 10 with ReLU is a common Split-MNIST choice
    def __init__(self):
        super().__init__()
        ...

    def forward(self, x):
        ...

## 5. Training and evaluation

In [ ]:
def train_one_task(model, dataset, epochs=EPOCHS_PER_TASK):
    """Plain training on this task's data only. No replay or regularization."""
    # TODO
    raise NotImplementedError


@torch.no_grad()
def evaluate(model, dataset):
    """Return accuracy in [0, 1]."""
    # TODO
    raise NotImplementedError

## 6. Learning the tasks in sequence

We record an accuracy matrix $R$, where $R_{i,j}$ is the test accuracy on task
$j$ after training on task $i$. Every later chapter reports this same matrix.

% TODO: Explain how to read it: the diagonal is "just learned", entries below the
% diagonal are "remembered", and entries above are "not seen yet".

In [ ]:
n_tasks = len(TASKS)
R = np.zeros((n_tasks, n_tasks))  # R[i, j] = accuracy on task j after training on task i

# model = MLP().to(device)
# for i, (train_ds, _) in enumerate(task_data):
#     train_one_task(model, train_ds)
#     for j, (_, test_ds) in enumerate(task_data):
#         R[i, j] = evaluate(model, test_ds)

## 7. Seeing the forgetting

In [ ]:
# TODO: Heatmap of R (tasks trained on the y-axis, tasks evaluated on the x-axis),
# with each cell annotated with its accuracy.

In [ ]:
# TODO: One line per task: accuracy on task j (y) vs. number of tasks trained so far (x).
# This is usually the plot that makes forgetting "click" for readers.

## 8. Putting numbers on it

For $T$ tasks:

$$
\text{ACC} = \frac{1}{T}\sum_{j=1}^{T} R_{T,j}
\qquad
\text{BWT} = \frac{1}{T-1}\sum_{j=1}^{T-1} \left(R_{T,j} - R_{j,j}\right)
$$

% TODO: Explain each in a sentence. A negative BWT means forgetting.
% Mention forward transfer (FWT) briefly and save it for a later chapter.

In [ ]:
def average_accuracy(R):
    # TODO
    raise NotImplementedError


def backward_transfer(R):
    # TODO
    raise NotImplementedError

## 9. Baseline: joint training

% TODO: Train one fresh model on all tasks at once (shuffled together) and report
% its accuracy. This is the usual upper bound: the "no forgetting" reference.

In [ ]:
# TODO: joint training baseline

## 10. What's actually going on?

% TODO: Pick one or two small experiments to look inside the model:
% - (single head) After each task, look at the output logits / biases. Does the model
%   just predict the most recent classes?
% - How far do the weights move from the task 1 solution as later tasks are trained?
% - Does more epochs per task make forgetting worse? Does a lower learning rate help?

In [ ]:
# TODO: your exploration

## Takeaways

% TODO: 3-4 bullet summary in your own words.

## Try it yourself

% TODO: 2-3 exercises, e.g.
% - Switch the output-head setup and compare BWT
% - Replace Split-MNIST with Permuted MNIST
% - Change the task order. Does forgetting depend on it?

## What's next

% TODO: Tease chapter 2. The output-head choice in section 3 turns out to define three
% different problems: task-, domain- and class-incremental learning.